In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import random
from scipy import stats
from localcider.sequenceParameters import SequenceParameters

MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
import matplotlib
from adjustText import adjust_text
plt.rcParams["font.family"] = "Arial"
from sklearn import cluster, datasets, mixture
from scipy.stats import chi2_contingency
from adjustText import adjust_text
names = pd.read_excel('../response_reviewers/20251017_IDRmdSAM_Expession_local.xlsx',sheet_name='names') 
from adjustText import adjust_text
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
import scipy
scipy.__version__

In [ ]:
df_cells = pd.read_csv('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/RAW/U20S/Cells.csv',delimiter=',')
df_puncta = pd.read_csv('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/RAW/U20S/puncta.csv',delimiter=',',header=1)
puncta = df_puncta.loc[:,['FileName_OrigImg','Parent_Cells','AreaShape_Area','AreaShape_Eccentricity','Intensity_IntegratedIntensity_MaskedPuncta']]
cells = df_cells.loc[:,['ObjectNumber','FileName_OrigImg','AreaShape_Area']]
l =[]
for file in tqdm(set(puncta.FileName_OrigImg)):
    element = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[0] # parent 
    topology = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[1]
    rep = file.split('_')[-2]
    for cell in set(puncta[puncta.FileName_OrigImg==file].Parent_Cells):
        number_puncta = len(puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)])
        area_puncta = puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].AreaShape_Area.sum()
        area = cells[(cells.FileName_OrigImg==file)&(cells.ObjectNumber==cell)].AreaShape_Area.values[0]
        puncta_inten = puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()

        l.append({'Protein':element,'ele':topology,'value':number_puncta/area,'cell':cell,'file':file,
                  'number_puncta':number_puncta,'area':(area),'rep':rep,'area_puncta':area_puncta,'puncta_inten':puncta_inten})
df  = pd.DataFrame(l)
df = df[df.area>np.percentile(df.area,20)]
df = df.reset_index()
df

In [ ]:
df['cell_type'] = 'u2os'

In [ ]:
df_cells = pd.read_csv('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/Cells.csv',delimiter=',')
df_puncta = pd.read_csv('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/puncta.csv',delimiter=',',header=1)
puncta = df_puncta.loc[:,['FileName_OrigImg','Parent_Cells','AreaShape_Area','AreaShape_Eccentricity','Intensity_IntegratedIntensity_MaskedPuncta','AreaShape_ConvexArea']]
cells = df_cells.loc[:,['ObjectNumber','FileName_OrigImg','AreaShape_Area']]
Eccentricity = .65
l =[]
for file in tqdm(set(puncta.FileName_OrigImg)):
    element = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[0] # parent 
    topology = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[1]
    rep = file.split('_')[-2]
    for cell in set(puncta[puncta.FileName_OrigImg==file].Parent_Cells):
        number_puncta = len(puncta[(puncta.AreaShape_Eccentricity<Eccentricity)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)])
        area_puncta = puncta[(puncta.AreaShape_Eccentricity<Eccentricity)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].AreaShape_Area.sum()
        Intensity = puncta[(puncta.AreaShape_Eccentricity<Eccentricity)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()
        area = cells[(cells.FileName_OrigImg==file)&(cells.ObjectNumber==cell)].AreaShape_Area.values[0]
        l.append({'Protein':element,'ele':topology,'value':number_puncta/area,'cell':cell,'file':file,
                  'number_puncta':number_puncta,'area':(area),'rep':rep,'area_puncta':area_puncta,'puncta_inten':Intensity})
df_u2os  = pd.DataFrame(l)
df_u2os = df_u2os[df_u2os.area>np.percentile(df_u2os.area,20)]
df_u2os = df_u2os.reset_index()
df_u2os['cell_type'] = 'hek'

In [ ]:
df_u2os['cell_type'] = 'hek'

In [ ]:
cell_counter_dSAM = 1
cell_counter_no_dSAM = 1
l = []
cell_df = pd.read_csv('//Users/aelazar/notebooks_for_paper/TRIF_data/APP2P/Cellprofiler_output/Cells.csv')
c = cell_df.loc[:,['ObjectNumber','AreaShape_Area','FileName_OrigImg']]
c.loc[c[c['FileName_OrigImg'].str.find('APP_2P')!=-1].index,'mdSAM'] = '2P'
c.loc[c[c['FileName_OrigImg'].str.find('DELTA')!=-1].index,'mdSAM'] = '-'
c.loc[c[c['FileName_OrigImg'].str.find('dAPP')!=-1].index,'mdSAM'] = '-'
c.loc[c[pd.isna(c.mdSAM)].index,'mdSAM']= '+'
p = pd.read_csv('/Users/aelazar/notebooks_for_paper/TRIF_data/APP2P/Cellprofiler_output/puncta.csv',header=1)
p_df  = p.loc[:,['AreaShape_Area','AreaShape_Eccentricity','Parent_Cells','FileName_OrigImg','Intensity_IntegratedIntensity_MaskedPuncta']]
counter = 1
for cell in c[c.mdSAM=='+'].index:
    cell_area = c.loc[cell,'AreaShape_Area']
    sample = c.loc[cell,'FileName_OrigImg']
    ObjectNumber = c.loc[cell,'ObjectNumber']
    #name = c.loc[cell,'mdSAM']
    for puncta in p_df[(p_df.Parent_Cells==ObjectNumber)&(p_df.FileName_OrigImg==sample)].index:
        puncta_area = p_df.loc[puncta,'AreaShape_Area']
        Eccentricity = p_df.loc[puncta,'AreaShape_Eccentricity']
        Intensity = p_df[(p_df.AreaShape_Eccentricity<Eccentricity)&(p_df.FileName_OrigImg==sample)&(p_df.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()

        l.append({'Cell_number':counter,'Eccentricity':Eccentricity,'cell_area':cell_area,'puncta_area':puncta_area,'puncta_inten':puncta_inten,'mdSAM':'+','image_name':sample})
    counter = counter +1 
counter = 1
for cell in c[c.mdSAM=='-'].index:
    cell_area = c.loc[cell,'AreaShape_Area']
    ObjectNumber = c.loc[cell,'ObjectNumber']
    sample = c.loc[cell,'FileName_OrigImg']
    #name = c.loc[cell,'mdSAM']
    for puncta in p_df[(p_df.Parent_Cells==ObjectNumber)&(p_df.FileName_OrigImg==sample)].index:
        puncta_area = p_df.loc[puncta,'AreaShape_Area']
        Eccentricity = p_df.loc[puncta,'AreaShape_Eccentricity']
        Intensity = p_df[(p_df.AreaShape_Eccentricity<Eccentricity)&(p_df.FileName_OrigImg==sample)&(p_df.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()

        l.append({'Cell_number':counter,'Eccentricity':Eccentricity,'cell_area':cell_area,'puncta_area':puncta_area,'mdSAM':'-','puncta_inten':puncta_inten,'image_name':sample,})
    counter = counter +1 
counter = 1
for cell in c[c.mdSAM=='2P'].index:
    cell_area = c.loc[cell,'AreaShape_Area']
    ObjectNumber = c.loc[cell,'ObjectNumber']
    sample = c.loc[cell,'FileName_OrigImg']
    #name = c.loc[cell,'mdSAM']
    for puncta in p_df[(p_df.Parent_Cells==ObjectNumber)&(p_df.FileName_OrigImg==sample)].index:
        puncta_area = p_df.loc[puncta,'AreaShape_Area']
        Eccentricity = p_df.loc[puncta,'AreaShape_Eccentricity']
        Intensity = p_df[(p_df.AreaShape_Eccentricity<Eccentricity)&(p_df.FileName_OrigImg==sample)&(p_df.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()

        l.append({'Cell_number':counter,'Eccentricity':Eccentricity,'cell_area':cell_area,'puncta_area':puncta_area,'mdSAM':'2P','puncta_inten':puncta_inten,'image_name':sample})
    counter = counter +1 

df_app = pd.DataFrame(l)
df_app['file_name']  = [i.split('-')[0] for i in df_app.image_name]
df_app.loc[df_app[df_app.file_name=='030322.sld '].index,'rep'] = 1
df_app.loc[df_app[df_app.file_name=='20250320_APP_TIRF.sld '].index,'rep'] = 1
df_app.loc[df_app[df_app.file_name.str.find('_BM.tif')!=-1].index,'rep'] = 2
df_app.loc[df_app[df_app.file_name.str.find('_BM_redo.tif')!=-1].index,'rep'] = 2


Eccentricity = .65
l =[]
mdSAM = ['-','+','2P']
for j in mdSAM:
    for i in set(df_app[df_app.mdSAM=='%s'%j].Cell_number):
        if df_app[(df_app.Cell_number==i)&(df_app.Eccentricity.between(.1,1))&(df_app.mdSAM=='%s'%j)].empty==False:
            rep = df_app[(df_app.Cell_number==i)&(df_app.mdSAM=='%s'%j)&(df_app.Eccentricity.between(.0,1))]['rep'].values[0]
            total_pucta_number = df_app[(df_app.Cell_number==i)&(df_app.mdSAM=='%s'%j)&(df_app.Eccentricity.between(.0,.65))].puncta_area.count()
            total_pucta_area = df_app[(df_app.Cell_number==i)&(df_app.mdSAM=='%s'%j)&(df_app.Eccentricity.between(.0,.65))].puncta_area.sum()
            cell_area = df_app[(df_app.Cell_number==i)&(df_app.mdSAM=='%s'%j)].cell_area.values[0]
            if j == '+':
                mdSAM_ = 'WT'
            if j == '2P':
                mdSAM_ = 'Mut'
            if j == '-':
                mdSAM_ = 'DEL'
                
            #print(total_pucta_area/cell_area)
            l.append({'Protein':'APP','ele':mdSAM_,'value':total_pucta_number/cell_area,'area':cell_area,'number_puncta':total_pucta_number,'rep':rep,'cell_type':'hek'})
        else:
            l.append({'Protein':'APP','ele':mdSAM_,'value':0,'area':cell_area,'number_puncta':total_pucta_number,'rep':rep,'cell_type':'hek'})
df_punta_cell = pd.DataFrame(l)
# df_punta_cell['box']='A'
# df_punta_cell
# df_punta_cell['psedo'] = df_punta_cell['number_pucta_cell_area']
# df_punta_cell['log_val'] =1*np.log10(df_punta_cell['psedo'])
# df_punta_cell['psedo_area'] = df_punta_cell['pucta_area_cell_area']
# df_punta_cell['log_val_area'] =1*np.log10(df_punta_cell['psedo_area'])



In [ ]:
np.sqrt(number_puncta / area)

In [ ]:
df_all = pd.concat([df,df_u2os,df_punta_cell])

In [ ]:
df_all['area_log'] = np.log(df_all['area'])
df_all['val_log']  = np.log1p(df_all.number_puncta/df_all.area)
df_all

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
name = 'A4'
# keep data at replicate level (DO NOT GROUP BY ele only)
# del df2
df2 = df_all.copy()
df2 = df2[(df2.cell_type=='u2os')&(df2.Protein == name)&(df2.ele!='DEL')]
df2 = df2.reset_index(drop=True)
df2['is_wt'] = (df2.ele == 'WT').astype(int)

model = sm.GLM(
    df2['number_puncta'],
    sm.add_constant(df2['is_wt']),
    family=sm.families.Poisson(),
    offset=np.log(df2['area'])
)

res = model.fit(cov_type='nonrobust')
print(res.summary())
beta = res.params['is_wt']
#beta = res.params["is_mut"]
se = res.bse["is_wt"]
p = res.pvalues["is_wt"]

rate_ratio = np.exp(beta)
ci_low = np.exp(beta - 1.96 * se)
ci_high = np.exp(beta + 1.96 * se)

print(f"Rate ratio (WT/mut) = {rate_ratio:.2f} [{ci_low:.2f}, {ci_high:.2f}]")
print(f"95% CI = [{ci_low:.2f}, {ci_high:.2f}]")
print(f"p-value = {p:.3g}")
beta = res.params["is_wt"]
p_two = res.pvalues["is_wt"]

# Expect WT > mut  ⇒ beta < 0
if beta > 0:
    p_one = p_two / 2
else:
    p_one = 1 - p_two / 2

print("one-sided p-value (WT > Mut):", p_one)

fig, ax = plt.subplots(figsize=(3,2))
#sns.boxplot(y='value',data=df[(df.Protein==name)&(df.area>min_)],x='ele',order=['WT','Mut'],dodge=True,fill=False)
sns.stripplot(y='cell_type',x='val_log',data=df_all[df_all.Protein==name],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut'],palette=['#C03B30','#AEAEAF'],dodge=True,alpha=0.2)
sns.boxplot(y='cell_type',x='val_log',data=df_all[df_all.Protein==name],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut'],palette=['#C03B30','#AEAEAF'],fill=False,    showcaps=False,    # removes the caps
    showfliers=False)
plt.legend([],frameon=False)
plt.locator_params(axis='x',nbins=3)
label = (
    f"RR = {rate_ratio:.2f} "
    f"[{ci_low:.2f}, {ci_high:.2f}]\n"
    f"p = {p:.3g}"
)

ax.text(
    0.5, 0.95, label,
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=8
)

In [ ]:
fig, ax = plt.subplots(figsize=(4,2))
#sns.boxplot(y='value',data=df[(df.Protein==name)&(df.area>min_)],x='ele',order=['WT','Mut'],dodge=True,fill=False)
sns.stripplot(y='cell_type',x='val_log',data=df_all[df_all.Protein=='APP'],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut','DEL'],palette=['#C03B30','#AEAEAF','#59595C'],dodge=True,alpha=0.2)
sns.boxplot(y='cell_type',x='val_log',data=df_all[df_all.Protein=='APP'],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut','DEL'],palette=['#C03B30','#AEAEAF','#59595C'],fill=False,    showcaps=False,    # removes the caps
    showfliers=False)
plt.legend([],frameon=False)
plt.ylabel('')

plt.locator_params(axis='x',nbins=3)
#plt.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/APP_Poisson.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
fig, ax = plt.subplots(figsize=(3,2))
#sns.boxplot(y='value',data=df[(df.Protein==name)&(df.area>min_)],x='ele',order=['WT','Mut'],dodge=True,fill=False)
sns.stripplot(y='cell_type',x='val_log',data=df_all[df_all.Protein==name],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut'],palette=['#C03B30','#AEAEAF'],dodge=True,alpha=0.2)
sns.boxplot(y='cell_type',x='val_log',data=df_all[df_all.Protein==name],order=['hek','u2os'],hue='ele',hue_order=['WT','Mut'],palette=['#C03B30','#AEAEAF'],fill=False,    showcaps=False,    # removes the caps
    showfliers=False)
plt.legend([],frameon=False)
#plt.locator_params(axis='x',nbins=3)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

def glm_gold_standard_figure(
    df_all: pd.DataFrame,
    protein: str,
    cell_types=("hek", "u2os"),
    wt_label="WT",
    mut_label="Mut",
    ele_col="ele",
    celltype_col="cell_type",
    protein_col="Protein",
    count_col="number_puncta",
    area_col="area",
    raw_x_col="val_log",
    exclude_eles=("DEL",),
    cov_type="nonrobust",
    figsize=(6, 3),   # slightly narrower overall
):
    """
    Gold-standard figure:
      Panel A: raw single-cell data (strip + box)
      Panel B: forest plot of RR (WT/Mut) with 95% CI
    """

    palette = ['#C03B30', '#AEAEAE']
    hue_order = [wt_label, mut_label]

    # -------------------------
    # Filter once
    # -------------------------
    df = df_all.copy()
    df = df[(df[protein_col] == protein) & (~df[ele_col].isin(exclude_eles))]
    df = df[df[ele_col].isin(hue_order)]
    df = df.dropna(subset=[celltype_col, ele_col, count_col, area_col])
    df = df[df[area_col] > 0].copy()

    # -------------------------
    # Fit GLM per cell type
    # -------------------------
    rows = []
    for ct in cell_types:
        d = df[df[celltype_col] == ct].copy()

        n_wt = int((d[ele_col] == wt_label).sum())
        n_mut = int((d[ele_col] == mut_label).sum())

        if n_wt == 0 or n_mut == 0:
            rows.append({
                "cell_type": ct,
                "n_WT": n_wt,
                "n_Mut": n_mut,
                "RR": np.nan,
                "CI_low": np.nan,
                "CI_high": np.nan,
                "p": np.nan
            })
            continue

        d["is_wt"] = (d[ele_col] == wt_label).astype(int)

        model = sm.GLM(
            d[count_col],
            sm.add_constant(d["is_wt"]),
            family=sm.families.Poisson(),
            offset=np.log(d[area_col])
        )
        res = model.fit(cov_type=cov_type)

        beta = res.params["is_wt"]
        se = res.bse["is_wt"]
        p_two = res.pvalues["is_wt"]
        
        # one-sided p-value: H_A = WT > Mut (beta > 0)
        if beta > 0:
            p_one = p_two / 2
        else:
            p_one = 1 - p_two / 2
        
        rows.append({
            "cell_type": ct,
            "n_WT": n_wt,
            "n_Mut": n_mut,
            "RR": np.exp(beta),
            "CI_low": np.exp(beta - 1.96 * se),
            "CI_high": np.exp(beta + 1.96 * se),
            "p_two_sided": p_two,
            "p_one_sided": p_one
        })

    res_df = pd.DataFrame(rows)

    # -------------------------
    # Figure layout (RIGHT PANEL THINNER)
    # -------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        1, 2,
        width_ratios=[1.6, 0.375],   # <<< thinner right panel
        wspace=0.035
    )

    # -------------------------
    # Panel A: raw data (NO DESPINE)
    # -------------------------
    ax1 = fig.add_subplot(gs[0, 0])

    sns.stripplot(
        data=df, y=celltype_col, x=raw_x_col,
        order=cell_types,
        hue=ele_col, hue_order=hue_order,
        palette=palette,
        dodge=True, alpha=0.25, ax=ax1
    )
    sns.boxplot(
        data=df, y=celltype_col, x=raw_x_col,
        order=cell_types,
        hue=ele_col, hue_order=hue_order,
        palette=palette,
        dodge=True, fill=False,
        showcaps=False, showfliers=False,
        ax=ax1
    )

    ax1.set_title(f"{protein}")
    ax1.set_xlabel(raw_x_col)
    ax1.tick_params(axis='x', labelsize=12)
    ax1.set_ylabel("")
    ax1.set_xlabel("")
    ax1.legend([], frameon=False)
    ax1.locator_params(axis='x',nbins=4)

    # # n labels
    # for i, ct in enumerate(cell_types):
    #     r = res_df[res_df.cell_type == ct].iloc[0]
    #     ax1.text(
    #         0.98, i, f"WT n={r.n_WT}, Mut n={r.n_Mut}",
    #         transform=ax1.get_yaxis_transform(),
    #         ha="right", va="center",
    #         fontsize=8
    #     )

    # -------------------------
    # Panel B: bar plot of RR with 95% CI
    # -------------------------
    ax2 = fig.add_subplot(gs[0, 1])
    
    # y positions (top-to-bottom)
    y = np.arange(len(res_df))
    
    # bar heights are RR; error bars come from CI
    rr = res_df["RR"].to_numpy()
    ci_low = res_df["CI_low"].to_numpy()
    ci_high = res_df["CI_high"].to_numpy()
    
    # asymmetric error bars: [RR - low, high - RR]
    xerr = np.vstack([rr - ci_low, ci_high - rr])
    
    # handle missing rows safely
    mask = np.isfinite(rr) & np.isfinite(ci_low) & np.isfinite(ci_high)
    
    ax2.barh(
        y[mask],
        rr[mask],
        xerr=xerr[:, mask],
        height=0.55,
        color="gray",
        edgecolor="k",
        linewidth=0.8,
        capsize=0
    )
    
    ax2.axvline(1, ls="--", lw=1, color="k")
    
    ax2.set_yticks(y)
    ax2.set_yticklabels([
        f"n={nwt}\n\n n={nmut}"
        for nwt, nmut in zip(res_df.n_WT, res_df.n_Mut)
    ],fontsize=12)

    ax2.invert_yaxis()
    
    ax2.set_xlabel("RR(WT/-)", fontsize=12)
#    ax2.set_title("Rate Ratio (WT/Proline)", fontsize=8)
    
    # x-limits
    xmax = np.nanmax(ci_high) * 1.1
    ax2.set_xlim(left=0.01, right=xmax * 1.15)
    
    # p-values annotations (place near x=1 line)
    for i, p in enumerate(res_df.p_one_sided.values):
        if np.isfinite(p):
            ax2.text(.5, i, f"p={p:.3g}", va="center", fontsize=8)
    
    # optional: remove extra clutter
    #ax2.set_xlabel("")  # keep tight on small panel
    
    return fig, res_df


In [ ]:
res_table_df = pd.DataFrame()
for i in ['A4','APP','KCNE4','EVI2A','MANS4','X3CL1','SYCY1','SIGL6','TIMD4','CD34']:
    fig, res_table = glm_gold_standard_figure(df_all, protein=i)
    #fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/%s_Poisson.svg'%i,bbox_inches='tight',transparent=True,dpi=900)
    # print(res_table)
    res_table['pro'] = i
    res_table_df = pd.concat([res_table_df,res_table])

In [ ]:
sns.barplot(x='pro',y='RR',data=res_table_df[res_table_df.cell_type=='hek'].sort_values(by='RR',ascending=False))
plt.axhline(1)
#plt.ylim([0.7,2.1])

In [ ]:
res_table_df#[res_table_df.cell_type=='hek'].sort_values(by='RR',)

In [ ]:
res_table

In [ ]:
fig, res_table = glm_gold_standard_figure(df_all, protein='APP',mut_label='DEL',exclude_eles=('Mut',))
fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/APP_DEL_Poisson.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
len(set(df_all.file.dropna()))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

def glm_gold_standard_figure_clustered(
    df_all: pd.DataFrame,
    protein: str,
    cell_types=("hek", "u2os"),
    wt_label="WT",
    mut_label="Mut",
    ele_col="ele",
    celltype_col="cell_type",
    protein_col="Protein",
    count_col="number_puncta",
    area_col="area",
    raw_x_col="val_log",
    rep_col="rep",                 # <<< NEW
    exclude_eles=("DEL",),
    figsize=(6, 3),
):
    """
    Panel A: raw single-cell data
    Panel B: RR (WT/Mut) with 95% CI from Poisson GLM with cluster-robust SE by replicate.
    """

    palette = ['#C03B30', '#AEAEAF']
    hue_order = [wt_label, mut_label]

    # -------------------------
    # Filter once
    # -------------------------
    df = df_all.copy()
    df = df[(df[protein_col] == protein) & (~df[ele_col].isin(exclude_eles))]
    df = df[df[ele_col].isin(hue_order)]
    df = df.dropna(subset=[celltype_col, ele_col, count_col, area_col, rep_col])
    df = df[df[area_col] > 0].copy()

    # -------------------------
    # Fit GLM per cell type (clustered by replicate)
    # -------------------------
    rows = []
    for ct in cell_types:
        d = df[df[celltype_col] == ct].copy()

        n_wt = int((d[ele_col] == wt_label).sum())
        n_mut = int((d[ele_col] == mut_label).sum())

        # replicate counts (useful to sanity check)
        reps_wt = d.loc[d[ele_col] == wt_label, rep_col].nunique()
        reps_mut = d.loc[d[ele_col] == mut_label, rep_col].nunique()

        if n_wt == 0 or n_mut == 0:
            rows.append({
                "cell_type": ct,
                "n_WT": n_wt, "n_Mut": n_mut,
                "nrep_WT": reps_wt, "nrep_Mut": reps_mut,
                "RR": np.nan, "CI_low": np.nan, "CI_high": np.nan,
                "p_two_sided": np.nan, "p_one_sided": np.nan
            })
            continue

        d["is_wt"] = (d[ele_col] == wt_label).astype(int)

        model = sm.GLM(
            d[count_col],
            sm.add_constant(d["is_wt"]),
            family=sm.families.Poisson(),
            offset=np.log(d[area_col])
        )
        # gee = sm.GEE(
        # endog=d[count_col],
        # exog=sm.add_constant(d["is_wt"]),
        # groups=d[rep_col],
        # family=sm.families.Poisson(),
        # offset=np.log(d[area_col]),
        # cov_struct=sm.cov_struct.Exchangeable()
        #         )
        # res = gee.fit()
        # Cluster-robust covariance by replicate
        # Note: requires enough replicates to be meaningful (rule of thumb: >= ~10 clusters).
        res = model.fit(cov_type="cluster", cov_kwds={"groups": d[rep_col]})

        beta = res.params["is_wt"]
        se = res.bse["is_wt"]
        p_two = res.pvalues["is_wt"]

        # one-sided p-value: H_A = WT > Mut (beta > 0)
        p_one = (p_two / 2) if beta > 0 else (1 - p_two / 2)

        rows.append({
            "cell_type": ct,
            "n_WT": n_wt, "n_Mut": n_mut,
            "nrep_WT": reps_wt, "nrep_Mut": reps_mut,
            "RR": float(np.exp(beta)),
            "CI_low": float(np.exp(beta - 1.96 * se)),
            "CI_high": float(np.exp(beta + 1.96 * se)),
            "p_two_sided": float(p_two),
            "p_one_sided": float(p_one),
        })

    res_df = pd.DataFrame(rows)

    # -------------------------
    # Figure layout
    # -------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.6, 0.375], wspace=0.035)

    # Panel A
    ax1 = fig.add_subplot(gs[0, 0])
    sns.stripplot(
        data=df, y=celltype_col, x=raw_x_col,
        order=cell_types,
        hue=ele_col, hue_order=hue_order,
        palette=palette, dodge=True, alpha=0.25, ax=ax1
    )
    sns.boxplot(
        data=df, y=celltype_col, x=raw_x_col,
        order=cell_types,
        hue=ele_col, hue_order=hue_order,
        palette=palette, dodge=True, fill=False,
        showcaps=False, showfliers=False, ax=ax1
    )
    ax1.set_title(f"{protein}")
    ax1.set_xlabel(raw_x_col)
    ax1.set_ylabel("")
    ax1.legend([], frameon=False)
    ax1.locator_params(axis='x', nbins=4)

    # Panel B
    ax2 = fig.add_subplot(gs[0, 1])

    y = np.arange(len(res_df))
    rr = res_df["RR"].to_numpy()
    ci_low = res_df["CI_low"].to_numpy()
    ci_high = res_df["CI_high"].to_numpy()

    xerr = np.vstack([rr - ci_low, ci_high - rr])
    mask = np.isfinite(rr) & np.isfinite(ci_low) & np.isfinite(ci_high)

    ax2.barh(
        y[mask], rr[mask],
        xerr=xerr[:, mask],
        height=0.55,
        color="gray", edgecolor="k", linewidth=0.8,
        capsize=0
    )
    ax2.axvline(1, ls="--", lw=1, color="k")

    ax2.set_yticks(y)
    ax2.set_yticklabels([
        f"n={nwt}\n\n n={nmut}"
        for nwt, nmut, rwt, rmut in zip(res_df.n_WT, res_df.n_Mut, res_df.nrep_WT, res_df.nrep_Mut)
    ])
    ax2.invert_yaxis()
    ax2.set_title("Rate Ratio (WT/Mut)\nPoisson GLM (clustered by rep)", fontsize=8)

    xmax = np.nanmax(ci_high) * 1.1 if np.isfinite(np.nanmax(ci_high)) else 2
    ax2.set_xlim(left=0.01, right=xmax * 1.15)

    for i, p in enumerate(res_df.p_one_sided.values):
        if np.isfinite(p):
            ax2.text(1, i, f"p={p:.3g}", va="center", fontsize=8)

    ax2.set_xlabel("")

    return fig, res_df


In [ ]:
for i in ['A4','APP','KCNE4','EVI2A','MANS4','X3CL1','SYCY1','SIGL6','TIMD4','CD34']:
    fig, res_table = glm_gold_standard_figure_clustered(df_all, protein=i)
    fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/%s_Poisson.svg'%i,bbox_inches='tight',transparent=True,dpi=900)
    # print(res_table)

In [ ]:
df_cells_HTR7 = pd.read_csv('/Users/aelazar/Downloads/HTR7_Vikas/Cells.csv',delimiter=',')
df_puncta_HTR7 = pd.read_csv('/Users/aelazar/Downloads/HTR7_Vikas/puncta.csv',delimiter=',',header=1)
puncta_HTR7 = df_puncta_HTR7.loc[:,['FileName_OrigImg','Parent_Cells','AreaShape_Area','AreaShape_Eccentricity','Intensity_IntegratedIntensity_MaskedPuncta']]
cells_HTR7 = df_cells_HTR7.loc[:,['ObjectNumber','FileName_OrigImg','AreaShape_Area']]
l =[]
for file in tqdm(set(puncta_HTR7.FileName_OrigImg)):
    if file.find('WT')==-1:
        element = 'MUT'
    else:
        element = 'WT'
    #element = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[0] # parent 
    topology = 'in'#names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[1]
    rep = int(file.split('_')[2].split('.')[0])
    for cell in set(puncta_HTR7[puncta_HTR7.FileName_OrigImg==file].Parent_Cells):
        number_puncta = len(puncta_HTR7[(puncta_HTR7.AreaShape_Eccentricity<.65)&(puncta_HTR7.FileName_OrigImg==file)&(puncta_HTR7.Parent_Cells==cell)])
        area_puncta = puncta_HTR7[(puncta_HTR7.AreaShape_Eccentricity<.65)&(puncta_HTR7.FileName_OrigImg==file)&(puncta_HTR7.Parent_Cells==cell)].AreaShape_Area.sum()
        puncta_inten = puncta_HTR7[(puncta_HTR7.AreaShape_Eccentricity<.65)&(puncta_HTR7.FileName_OrigImg==file)&(puncta_HTR7.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()
        area = cells_HTR7[(cells_HTR7.FileName_OrigImg==file)&(cells_HTR7.ObjectNumber==cell)].AreaShape_Area.values[0]
        l.append({'Protein':element,'ele':topology,'value':number_puncta/area,'cell':cell,'file':file,
                  'number_puncta':number_puncta,'area':(area),'rep':rep,'area_puncta':area_puncta,'Puncta_Intensity':puncta_inten})
df_HTR7  = pd.DataFrame(l)
df_HTR7 = df_HTR7[df_HTR7.area>np.percentile(df_HTR7.area,20)]
df_HTR7 = df_HTR7.reset_index()
df_HTR7

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns


from typing import Optional

def glm_gold_standard_figure_clustered_single_group(
    df_all: pd.DataFrame,
    title: str = "HTR7",
    wt_label: str = "WT",
    mut_label: str = "MUT",
    condition_col: str = "Protein",
    count_col: str = "number_puncta",
    area_col: str = "area",
    raw_x_col: str = "value",
    rep_col: str = "rep",
    ele_col: str = "ele",
    include_ele: Optional[str] = "in",
    figsize=(5.5, 2.4),
):
    """
    Panel A: raw single-cell data
    Panel B: RR (WT/MUT) with 95% CI from Poisson GLM,
             using cluster-robust SE by replicate.

    Adapted for dataframe with columns:
    Protein, ele, value, number_puncta, area, rep, etc.

    Assumes:
        Protein == WT or MUT
        value == raw density-like value to plot
        number_puncta == count outcome
        area == exposure / offset
        rep == replicate cluster
    """

    palette = ["#C03B30", "#AEAEAF"]
    hue_order = [wt_label, mut_label]

    # -------------------------
    # Filter data
    # -------------------------
    df = df_all.copy()

    if include_ele is not None and ele_col in df.columns:
        df = df[df[ele_col] == include_ele].copy()

    df = df[df[condition_col].isin(hue_order)].copy()

    df = df.dropna(
        subset=[
            condition_col,
            count_col,
            area_col,
            raw_x_col,
            rep_col,
        ]
    ).copy()

    df = df[df[area_col] > 0].copy()

    # Make sure counts are numeric integers/floats
    df[count_col] = pd.to_numeric(df[count_col], errors="coerce")
    df[area_col] = pd.to_numeric(df[area_col], errors="coerce")
    df[raw_x_col] = pd.to_numeric(df[raw_x_col], errors="coerce")

    df = df.dropna(subset=[count_col, area_col, raw_x_col]).copy()

    # -------------------------
    # Fit Poisson GLM
    # -------------------------
    n_wt = int((df[condition_col] == wt_label).sum())
    n_mut = int((df[condition_col] == mut_label).sum())

    reps_wt = df.loc[df[condition_col] == wt_label, rep_col].nunique()
    reps_mut = df.loc[df[condition_col] == mut_label, rep_col].nunique()

    if n_wt == 0 or n_mut == 0:
        res_df = pd.DataFrame(
            [
                {
                    "group": title,
                    "n_WT": n_wt,
                    "n_MUT": n_mut,
                    "nrep_WT": reps_wt,
                    "nrep_MUT": reps_mut,
                    "RR": np.nan,
                    "CI_low": np.nan,
                    "CI_high": np.nan,
                    "p_two_sided": np.nan,
                    "p_one_sided": np.nan,
                }
            ]
        )

    else:
        df["is_wt"] = (df[condition_col] == wt_label).astype(int)

        model = sm.GLM(
            df[count_col],
            sm.add_constant(df["is_wt"]),
            family=sm.families.Poisson(),
            offset=np.log(df[area_col]),
        )

        # Cluster-robust covariance by replicate
        res = model.fit(
            cov_type="cluster",
            cov_kwds={"groups": df[rep_col]},
        )

        beta = res.params["is_wt"]
        se = res.bse["is_wt"]
        p_two = res.pvalues["is_wt"]

        # One-sided p-value: H_A = WT > MUT
        p_one = (p_two / 2) if beta > 0 else (1 - p_two / 2)

        res_df = pd.DataFrame(
            [
                {
                    "group": title,
                    "n_WT": n_wt,
                    "n_MUT": n_mut,
                    "nrep_WT": reps_wt,
                    "nrep_MUT": reps_mut,
                    "RR": float(np.exp(beta)),
                    "CI_low": float(np.exp(beta - 1.96 * se)),
                    "CI_high": float(np.exp(beta + 1.96 * se)),
                    "p_two_sided": float(p_two),
                    "p_one_sided": float(p_one),
                }
            ]
        )

    # -------------------------
    # Figure layout
    # -------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.6, 0.55],
        wspace=0.08,
    )

    # -------------------------
    # Panel A: raw data
    # -------------------------
    ax1 = fig.add_subplot(gs[0, 0])

    sns.stripplot(
        data=df,
        y=np.repeat(title, len(df)),
        x=raw_x_col,
        hue=condition_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        alpha=0.35,
        ax=ax1,
    )

    sns.boxplot(
        data=df,
        y=np.repeat(title, len(df)),
        x=raw_x_col,
        hue=condition_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        fill=False,
        showcaps=False,
        showfliers=False,
        ax=ax1,
    )

    ax1.set_title(title)
    ax1.set_xlabel(raw_x_col)
    ax1.set_ylabel("")
    ax1.legend([], frameon=False)
    ax1.locator_params(axis="x", nbins=4)

    # -------------------------
    # Panel B: rate ratio
    # -------------------------
    ax2 = fig.add_subplot(gs[0, 1])

    rr = res_df.loc[0, "RR"]
    ci_low = res_df.loc[0, "CI_low"]
    ci_high = res_df.loc[0, "CI_high"]

    if np.isfinite(rr) and np.isfinite(ci_low) and np.isfinite(ci_high):
        xerr = np.array([[rr - ci_low], [ci_high - rr]])

        ax2.barh(
            [0],
            [rr],
            xerr=xerr,
            height=0.55,
            color="gray",
            edgecolor="k",
            linewidth=0.8,
            capsize=0,
        )

    ax2.axvline(1, ls="--", lw=1, color="k")

    ax2.set_yticks([0])
    ax2.set_yticklabels(
        [
            f"WT n={n_wt}\nMUT n={n_mut}"
        ]
    )

    ax2.set_title(
        "Rate Ratio\nWT/MUT",
        fontsize=8,
    )

    if np.isfinite(ci_high):
        xmax = ci_high * 1.25
    else:
        xmax = 2

    ax2.set_xlim(left=0.01, right=max(1.5, xmax))

    p = res_df.loc[0, "p_one_sided"]
    if np.isfinite(p):
        ax2.text(
            1,
            0,
            f"p={p:.3g}",
            va="center",
            ha="left",
            fontsize=8,
        )

    ax2.set_xlabel("")
    ax2.invert_yaxis()

    plt.tight_layout()

    return fig, res_df

In [ ]:
fig, res_df = glm_gold_standard_figure_clustered_single_group(
    df_HTR7,
    title="HTR7",
    wt_label="WT",
    mut_label="MUT",
    condition_col="Protein",
    count_col="number_puncta",
    area_col="area",
    raw_x_col="value",
    rep_col="rep",
    include_ele="in",
)

res_df
fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/HR7_TIRF.svg',bbox_inches='tight',transparent=True,dpi=900)

In [ ]:
res_df

In [ ]:
sns.ecdfplot(x='Puncta_Intensity',data=df_HTR7,hue='Protein')
df_HTR7

In [ ]:
xx = (
    df_HTR7
    .groupby(["Protein", "rep"])["value"]
    .quantile(0.50)
    .reset_index(name="top_quartile_value")
)
xx

In [ ]:
sns.stripplot(y='top_quartile_value',data=xx,x='Protein',hue='rep')

In [ ]:
stats.ttest_rel(xx[xx.Protein=='WT'].top_quartile_value,xx[xx.Protein!='WT'].top_quartile_value,alternative='greater')

In [ ]:
sns.violinplot(y='Puncta_Intensity',data=df_HTR7,hue='Protein',split=True,fill=False)

In [ ]:
df_HTR7['pi_vs_pi_area'] = df_HTR7['Puncta_Intensity']/df_HTR7['area_puncta']
df_HTR7['pi_vs_value'] = df_HTR7['Puncta_Intensity']/df_HTR7['value']


In [ ]:
sns.boxplot(y='pi_vs_pi_area',data=df_HTR7,hue='Protein',fill=False)

In [ ]:
sns.boxplot(y='pi_vs_value',data=df_HTR7,hue='Protein',fill=False)

In [ ]:
df_all['pi_vs_pi_area'] = df_all['puncta_inten']/df_all['area_puncta']
df_all['pi_vs_value'] = df_all['puncta_inten']/df_all['value']


In [ ]:
sns.boxplot(x='Protein',y='pi_vs_pi_area',data=df_all[df_all.Protein=='X3CL1'],hue='ele',fill=False)

In [ ]:
df_all[df_all.Protein=='X3CL1'].groupby(['ele']).pi_vs_value.describe()

In [ ]:
sns.boxplot(hue='ele',y='pi_vs_pi_area',data=df_all[(df_all.cell_type!='u2os')].reset_index(drop=True),x='Protein')
plt.yscale('log')

In [ ]:
df_all.drop_duplicates('Protein')

In [ ]:

df_cells = pd.read_csv('/Users/aelazar/Downloads/TIFFs/Cells.csv',delimiter=',')
df_puncta = pd.read_csv('/Users/aelazar/Downloads/TIFFs/puncta.csv',delimiter=',',header=1)
puncta = df_puncta.loc[:,['FileName_OrigImg','Parent_Cells','AreaShape_Area','AreaShape_Eccentricity','Intensity_IntegratedIntensity_MaskedPuncta']]
cells = df_cells.loc[:,['ObjectNumber','FileName_OrigImg','AreaShape_Area']]
l =[]
for file in tqdm(set(puncta.FileName_OrigImg)):
    #element = names[names.well==file.split('_')[-3].split('-')[-1][1:]].name.values[0].split('_')[0] # parent 
    #cell_type = 
    topology= file.split('_')[-3]
    element = file.split('_')[-4]
    cell_type = file.split('_')[-5]
    rep = file.split('_')[-2]
    for cell in set(puncta[puncta.FileName_OrigImg==file].Parent_Cells):
        number_puncta = len(puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)])
        area_puncta = puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].AreaShape_Area.sum()
        area = cells[(cells.FileName_OrigImg==file)&(cells.ObjectNumber==cell)].AreaShape_Area.values[0]
        puncta_inten = puncta[(puncta.AreaShape_Eccentricity<.65)&(puncta.FileName_OrigImg==file)&(puncta.Parent_Cells==cell)].Intensity_IntegratedIntensity_MaskedPuncta.sum()

        l.append({'Protein':element,'ele':topology,'value':number_puncta/area,'cell':cell,'file':file,'cell_type':cell_type,
                  'number_puncta':number_puncta,'area':(area),'rep':rep,'area_puncta':area_puncta,'puncta_inten':puncta_inten})
df  = pd.DataFrame(l)
df = df[df.area>np.percentile(df.area,20)]
df = df.reset_index()
df

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm
from typing import Sequence


def glm_gold_standard_figure_clustered_by_celltype(
    df_all: pd.DataFrame,
    protein: str,
    cell_types: Sequence[str] = ("hek", "u2os"),
    wt_label: str = "WT",
    mut_label: str = "MUT",
    ele_col: str = "ele",
    celltype_col: str = "cell_type",
    protein_col: str = "Protein",
    count_col: str = "number_puncta",
    area_col: str = "area",
    raw_x_col: str = "value",
    rep_col: str = "rep",
    exclude_eles: Sequence[str] = ("DEL",),
    figsize=(6, 3),
):
    """
    For one protein, compute WT/MUT rate ratio separately for each cell type.

    Model per cell type:

        number_puncta ~ is_wt + offset(log(area))

    Rate ratio:

        RR = WT / MUT

    One-sided hypothesis test:

        H_A: WT > MUT

    Uses cluster-robust SE by replicate when there is more than one replicate.
    """

    palette = {
        wt_label: "#C03B30",
        mut_label: "#AEAEAF",
    }

    hue_order = [wt_label, mut_label]

    # -------------------------
    # Filter once
    # -------------------------
    df = df_all.copy()

    df[count_col] = pd.to_numeric(df[count_col], errors="coerce")
    df[area_col] = pd.to_numeric(df[area_col], errors="coerce")
    df[raw_x_col] = pd.to_numeric(df[raw_x_col], errors="coerce")

    df = df[
        (df[protein_col] == protein)
        & (~df[ele_col].isin(exclude_eles))
        & (df[ele_col].isin(hue_order))
        & (df[celltype_col].isin(cell_types))
    ].copy()

    df = df.dropna(
        subset=[
            celltype_col,
            ele_col,
            count_col,
            area_col,
            raw_x_col,
            rep_col,
        ]
    ).copy()

    df = df[df[area_col] > 0].copy()

    if df.empty:
        raise ValueError(
            "No data left after filtering.\n\n"
            f"{protein_col} values:\n{df_all[protein_col].value_counts()}\n\n"
            f"{ele_col} values:\n{df_all[ele_col].value_counts()}\n\n"
            f"{celltype_col} values:\n{df_all[celltype_col].value_counts()}"
        )

    # -------------------------
    # Fit GLM per cell type
    # -------------------------
    rows = []

    for ct in cell_types:
        d = df[df[celltype_col] == ct].copy()

        n_wt = int((d[ele_col] == wt_label).sum())
        n_mut = int((d[ele_col] == mut_label).sum())

        reps_wt = d.loc[d[ele_col] == wt_label, rep_col].nunique()
        reps_mut = d.loc[d[ele_col] == mut_label, rep_col].nunique()
        total_reps = d[rep_col].nunique()

        if n_wt == 0 or n_mut == 0:
            rows.append(
                {
                    "protein": protein,
                    "cell_type": ct,
                    "n_WT": n_wt,
                    "n_MUT": n_mut,
                    "nrep_WT": reps_wt,
                    "nrep_MUT": reps_mut,
                    "RR_WT_over_MUT": np.nan,
                    "CI_low": np.nan,
                    "CI_high": np.nan,
                    "z_WT_gt_MUT": np.nan,
                    "p_one_sided_WT_gt_MUT": np.nan,
                }
            )
            continue

        d["is_wt"] = (d[ele_col] == wt_label).astype(int)

        model = sm.GLM(
            d[count_col],
            sm.add_constant(d["is_wt"]),
            family=sm.families.Poisson(),
            offset=np.log(d[area_col]),
        )

        if total_reps > 1:
            res = model.fit(
                cov_type="cluster",
                cov_kwds={"groups": d[rep_col]},
            )
        else:
            res = model.fit()

        beta = res.params["is_wt"]
        se = res.bse["is_wt"]

        z = beta / se

        # One-sided p-value for H_A: WT > MUT
        p_one = norm.sf(z)

        rows.append(
            {
                "protein": protein,
                "cell_type": ct,
                "n_WT": n_wt,
                "n_MUT": n_mut,
                "nrep_WT": reps_wt,
                "nrep_MUT": reps_mut,
                "RR_WT_over_MUT": float(np.exp(beta)),
                "CI_low": float(np.exp(beta - 1.96 * se)),
                "CI_high": float(np.exp(beta + 1.96 * se)),
                "z_WT_gt_MUT": float(z),
                "p_one_sided_WT_gt_MUT": float(p_one),
            }
        )

    res_df = pd.DataFrame(rows)

    # -------------------------
    # Figure layout
    # -------------------------
    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.6, 0.45],
        wspace=0.08,
    )

    # -------------------------
    # Panel A: raw data
    # -------------------------
    ax1 = fig.add_subplot(gs[0, 0])

    sns.stripplot(
        data=df,
        y=celltype_col,
        x=raw_x_col,
        order=list(cell_types),
        hue=ele_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        alpha=0.25,
        ax=ax1,
    )

    sns.boxplot(
        data=df,
        y=celltype_col,
        x=raw_x_col,
        order=list(cell_types),
        hue=ele_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        fill=False,
        showcaps=False,
        showfliers=False,
        ax=ax1,
    )

    ax1.set_title(protein)
    ax1.set_xlabel(raw_x_col)
    ax1.set_ylabel("")
    ax1.legend([], frameon=False)
    ax1.locator_params(axis="x", nbins=4)

    # -------------------------
    # Panel B: rate ratio
    # -------------------------
    ax2 = fig.add_subplot(gs[0, 1])

    y = np.arange(len(res_df))

    rr = res_df["RR_WT_over_MUT"].to_numpy()
    ci_low = res_df["CI_low"].to_numpy()
    ci_high = res_df["CI_high"].to_numpy()

    mask = np.isfinite(rr) & np.isfinite(ci_low) & np.isfinite(ci_high)

    if mask.any():
        xerr = np.vstack(
            [
                rr[mask] - ci_low[mask],
                ci_high[mask] - rr[mask],
            ]
        )

        ax2.barh(
            y[mask],
            rr[mask],
            xerr=xerr,
            height=0.55,
            color="gray",
            edgecolor="k",
            linewidth=0.8,
            capsize=0,
        )

    ax2.axvline(1, ls="--", lw=1, color="k")

    ax2.set_yticks(y)
    ax2.set_yticklabels(
        [
            f"{row['cell_type']}\nWT n={row['n_WT']}\nMUT n={row['n_MUT']}"
            for _, row in res_df.iterrows()
        ],
        fontsize=8,
    )

    ax2.invert_yaxis()

    ax2.set_title(
        "Rate Ratio\nWT/MUT",
        fontsize=8,
    )

    finite_ci_high = res_df["CI_high"].replace([np.inf, -np.inf], np.nan).dropna()

    if len(finite_ci_high) > 0:
        xmax = max(1.5, finite_ci_high.max() * 1.25)
    else:
        xmax = 2

    ax2.set_xlim(left=0.01, right=xmax)

    for i, p in enumerate(res_df["p_one_sided_WT_gt_MUT"].values):
        if np.isfinite(p):
            ax2.text(
                1,
                i,
                f"p={p:.3g}",
                va="center",
                ha="left",
                fontsize=8,
            )

    ax2.set_xlabel("")

    plt.tight_layout()

    return fig, res_df

In [ ]:
fig, res_df_HTR7 = glm_gold_standard_figure_clustered_by_celltype(
    df_all=df,
    protein="HTR7",
    cell_types=("HEK", "U2OS"),
    wt_label="WT",
    mut_label="MUT",
    ele_col="ele",
    celltype_col="cell_type",
    raw_x_col="value",
)

fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/SNAPtn_HTR7.svg',bbox_inches='tight',transparent=True,dpi=900)
res_df_HTR7

In [ ]:
fig, res_df_KCNE4 = glm_gold_standard_figure_clustered_by_celltype(
    df_all=df,
    protein="KCNE4",
    cell_types=("HEK", "U2OS"),
    wt_label="WT",
    mut_label="MUT",
    ele_col="ele",
    celltype_col="cell_type",
    raw_x_col="value",
)

fig.savefig('/Users/aelazar/notebooks_for_paper/response_reviewers/TRIF_data/SNAPtn_KCNE4.svg',bbox_inches='tight',transparent=True,dpi=900)

res_df_KCNE4

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns


def format_p_value(p: float) -> str:
    """Format p-values for publication-style annotation."""
    if not np.isfinite(p):
        return "NA"
    if p == 0:
        return "0"
    elif p < 1e-1:
        exp = int(f"{p:.1e}".split("e")[1])
        mantissa = p / (10**exp)
        return rf"{mantissa:.1f}\times10^{{{exp}}}"
    else:
        return f"{p:.3f}"


def glm_gold_standard_figure(
    df_all: pd.DataFrame,
    protein: str,
    cell_type="hek",
    wt_label="WT",
    mut_label="Mut",
    ele_col="ele",
    celltype_col="cell_type",
    protein_col="Protein",
    count_col="number_puncta",
    area_col="area",
    raw_x_col="val_log",
    exclude_eles=("DEL",),
    cov_type="nonrobust",
    figsize=(6, 2.2),
):
    """
    Create a HEK-only two-panel figure.

    Panel A:
        Raw single-cell data shown as strip plots and box plots.

    Panel B:
        Rate ratio, RR = WT / Mut, with 95% confidence interval.
        The RR and one-sided p-value are shown as annotations.

    The one-sided hypothesis is:
        H_A: WT > Mut
    """

    palette = ["#C03B30", "#AEAEAE"]
    hue_order = [wt_label, mut_label]

    # -------------------------
    # Filter to one protein and HEK cells only
    # -------------------------
    df = df_all.copy()

    df = df[
        (df[protein_col] == protein)
        & (df[celltype_col].str.lower() == cell_type.lower())
        & (~df[ele_col].isin(exclude_eles))
        & (df[ele_col].isin(hue_order))
    ].copy()

    df = df.dropna(
        subset=[
            celltype_col,
            ele_col,
            count_col,
            area_col,
            raw_x_col,
        ]
    )

    df = df[df[area_col] > 0].copy()

    # -------------------------
    # Fit Poisson GLM
    # -------------------------
    n_wt = int((df[ele_col] == wt_label).sum())
    n_mut = int((df[ele_col] == mut_label).sum())

    if n_wt == 0 or n_mut == 0:
        result = {
            "cell_type": cell_type,
            "n_WT": n_wt,
            "n_Mut": n_mut,
            "RR": np.nan,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "p_two_sided": np.nan,
            "p_one_sided": np.nan,
        }

    else:
        df["is_wt"] = (df[ele_col] == wt_label).astype(int)

        model = sm.GLM(
            endog=df[count_col],
            exog=sm.add_constant(df["is_wt"]),
            family=sm.families.Poisson(),
            offset=np.log(df[area_col]),
        )

        res = model.fit(cov_type=cov_type)

        beta = res.params["is_wt"]
        se = res.bse["is_wt"]
        p_two_sided = res.pvalues["is_wt"]

        # One-sided p-value for H_A: WT > Mut
        if beta > 0:
            p_one_sided = p_two_sided / 2
        else:
            p_one_sided = 1 - p_two_sided / 2

        result = {
            "cell_type": cell_type,
            "n_WT": n_wt,
            "n_Mut": n_mut,
            "RR": np.exp(beta),
            "CI_low": np.exp(beta - 1.96 * se),
            "CI_high": np.exp(beta + 1.96 * se),
            "p_two_sided": p_two_sided,
            "p_one_sided": p_one_sided,
        }

    res_df = pd.DataFrame([result])

    # -------------------------
    # Figure layout
    # -------------------------
    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.6, 0.55],
        wspace=0.08,
    )

    # -------------------------
    # Panel A: raw HEK-cell data
    # -------------------------
    ax1 = fig.add_subplot(gs[0, 0])

    sns.stripplot(
        data=df,
        y=celltype_col,
        x=raw_x_col,
        order=[cell_type],
        hue=ele_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        alpha=0.25,
        ax=ax1,
    )

    sns.boxplot(
        data=df,
        y=celltype_col,
        x=raw_x_col,
        order=[cell_type],
        hue=ele_col,
        hue_order=hue_order,
        palette=palette,
        dodge=True,
        fill=False,
        showcaps=False,
        showfliers=False,
        ax=ax1,
    )

    ax1.set_title(protein)
    ax1.set_xlabel("")
    ax1.set_ylabel("")
    ax1.set_yticklabels(["HEK"])
    ax1.tick_params(axis="x", labelsize=12)
    ax1.tick_params(axis="y", labelsize=12)
    ax1.locator_params(axis="x", nbins=4)
    ax1.legend([], [], frameon=False)

    # -------------------------
    # Panel B: RR with 95% CI
    # -------------------------
    ax2 = fig.add_subplot(gs[0, 1])

    rr = res_df.loc[0, "RR"]
    ci_low = res_df.loc[0, "CI_low"]
    ci_high = res_df.loc[0, "CI_high"]
    p_one_sided = res_df.loc[0, "p_one_sided"]

    if np.all(np.isfinite([rr, ci_low, ci_high])):
        xerr = np.array(
            [
                [rr - ci_low],
                [ci_high - rr],
            ]
        )

        ax2.barh(
            y=[0],
            width=[rr],
            xerr=xerr,
            height=0.55,
            color="gray",
            edgecolor="black",
            linewidth=0.8,
            capsize=0,
        )

        xmax = max(ci_high * 1.45, 1.5)
        ax2.set_xlim(0.01, xmax)

        # Publication-style annotation
        p_text = format_p_value(p_one_sided)

        annotation = (
            rf"$RR={rr:.2f}$"
            "\n"
            rf"$p={p_text}$"
        )

        annotation_x = min(
            ci_high + 0.04 * xmax,
            0.97 * xmax,
        )

        ax2.text(
            annotation_x,
            0,
            annotation,
            va="center",
            ha="left",
            fontsize=9,
        )

    else:
        ax2.set_xlim(0.01, 1.5)

        ax2.text(
            0.75,
            0,
            "Insufficient data",
            ha="center",
            va="center",
            fontsize=9,
        )

    ax2.axvline(
        1,
        linestyle="--",
        linewidth=1,
        color="black",
    )

    ax2.set_yticks([0])
    ax2.set_yticklabels(
        [f"WT n={n_wt}\nMut n={n_mut}"],
        fontsize=10,
    )

    ax2.set_xlabel("RR (WT/Mut)", fontsize=11)
    ax2.set_ylim(-0.6, 0.6)
    plt.savefig('SNAPtm_%s_hek.svg'%protein,bbox_inches='tight',transparent=False)
    return fig, res_df

In [ ]:
fig, res_df = glm_gold_standard_figure(df_all, "APP")

In [ ]:
df['val_log'] = df['value']

In [ ]:
fig, res_df = glm_gold_standard_figure(df, "KCNE4")

In [ ]:
df["cell_type"] = df["cell_type"].replace("HEK", "hek")

In [ ]:
df["ele"] = df["ele"].replace("MUT", "Mut")
fig, res_df = glm_gold_standard_figure(df, "HTR7")

In [ ]:
fig, res_df = glm_gold_standard_figure(df, "KCNE4")